# 🎬 ASTRAA — LTX-Video Colab
Free/open-source image-to-video test for ASTRAA.

## 1. Check GPU
Run this first. A free T4 is ideal for this lightweight test.

In [ ]:
!nvidia-smi


## 2. Install LTX-Video
Uses the official Lightricks repository.

In [ ]:
%cd /content
!rm -rf LTX-Video
!git clone --depth 1 https://github.com/Lightricks/LTX-Video.git
!cd /content/LTX-Video && git fetch --depth 1 origin bdc8f017f0148a0f0bb9e3a5049d2d356423cee0 && git checkout bdc8f017f0148a0f0bb9e3a5049d2d356423cee0
%cd /content/LTX-Video
!pip install -q -e '.[inference]'
!pip install -q --force-reinstall --no-deps 'huggingface-hub~=0.30'
!pip install -q 'accelerate>=0.26'


## 3. Download the lighter 2B distilled model
This is intended for lighter VRAM than the 13B model.

In [ ]:
from huggingface_hub import hf_hub_download
model_dir='/content/LTX-Video/models'
import os
os.makedirs(model_dir, exist_ok=True)
hf_hub_download(repo_id='Lightricks/LTX-Video', filename='ltxv-2b-0.9.8-distilled.safetensors', local_dir=model_dir)
print('Model downloaded:', model_dir)


## 4. Upload the ASTRAA reference image
Upload one image such as Aarav + Maa Meera. Keep the image in `/content/LTX-Video/`.

In [ ]:
from google.colab import files
uploaded=files.upload()
image_name=next(iter(uploaded))
image_path=f'/content/LTX-Video/{image_name}'
print(image_path)


## 5. Generate the first test shot
Start with an ultra-small 9-frame T4 test. This first run uses CPU offload to avoid a T4 runtime crash.


In [ ]:
import os, sys, glob, yaml, subprocess, re

BASE = "/content/LTX-Video"
MODULE = f"{BASE}/ltx_video/inference.py"
RUNNER = f"{BASE}/inference.py"
BASE_CONFIG = f"{BASE}/configs/ltxv-2b-0.9.8-distilled.yaml"
MODEL = f"{BASE}/models/ltxv-2b-0.9.8-distilled.safetensors"
LOCAL_CONFIG = f"{BASE}/configs/astraa-t4-safe.yaml"
OUTPUT = f"{BASE}/outputs/astraa_test"

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("ASTRAA Step 5")
print("Checking files...")
for p in [MODULE, RUNNER, BASE_CONFIG, MODEL]:
    print(p, os.path.isfile(p))
    if not os.path.isfile(p):
        raise FileNotFoundError(f"Missing required file: {p}. Run Steps 2-4 first.")

# Patch the official 0.9.8 startup without depending on exact indentation.
src = open(MODULE, encoding="utf-8").read()

if "transformer = transformer.to(device)" in src:
    src = src.replace("    transformer = transformer.to(device)\n", "", 1)
    src = src.replace("    vae = vae.to(device)\n", "", 1)
    src = src.replace("    text_encoder = text_encoder.to(device)\n", "", 1)
    print("Startup patch applied.")
elif "enable_model_cpu_offload(device=device)" in src:
    print("Startup patch already present.")
else:
    raise RuntimeError("Could not find the expected LTX 0.9.8 startup lines. Do not continue; send this error screenshot.")

old = "    pipeline = LTXVideoPipeline(**submodel_dict)\n    pipeline = pipeline.to(device)\n    return pipeline\n"
new = "    pipeline = LTXVideoPipeline(**submodel_dict)\n    if device == \"cuda\":\n        pipeline.enable_model_cpu_offload(device=device)\n    else:\n        pipeline = pipeline.to(device)\n    return pipeline\n"

if old in src:
    src = src.replace(old, new, 1)
    print("Pipeline offload patch applied.")
elif "pipeline.enable_model_cpu_offload(device=device)" in src:
    print("Pipeline offload patch already present.")
else:
    raise RuntimeError("Could not find the expected LTX pipeline block. Do not continue; send this error screenshot.")

open(MODULE, "w", encoding="utf-8").write(src)

with open(BASE_CONFIG, encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
cfg["checkpoint_path"] = MODEL
cfg.pop("pipeline_type", None)
cfg.pop("spatial_upscaler_model_path", None)
cfg["prompt_enhancement_words_threshold"] = 0
with open(LOCAL_CONFIG, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

images = []
for ext in ["*.jpg", "*.jpeg", "*.png", "*.webp"]:
    images.extend(glob.glob(f"{BASE}/{ext}"))
if not images:
    raise FileNotFoundError("No reference image found. Run Step 4 first.")
IMAGE = images[0]
os.makedirs(OUTPUT, exist_ok=True)

PROMPT = ("Aarav's mother Meera remains visually consistent with the reference image. "
          "She is inside her house at night. The curtains gently move in the wind. "
          "She slowly looks toward the entrance with a worried protective expression. "
          "Slow cinematic camera push-in, subtle dust particles, dramatic nighttime lighting, "
          "high-quality 3D animated Indian fantasy movie style, natural motion, no dialogue, no text.")

cmd = [sys.executable, RUNNER, "--prompt", PROMPT, "--output_path", OUTPUT,
       "--pipeline_config", LOCAL_CONFIG, "--conditioning_media_paths", IMAGE,
       "--conditioning_start_frames", "0", "--height", "128", "--width", "192",
       "--num_frames", "9", "--frame_rate", "24", "--seed", "42", "--offload_to_cpu"]

print("IMAGE:", IMAGE)
print("Starting generation...")
result = subprocess.run(cmd, cwd=BASE, capture_output=True, text=True)
print("\n===== STDOUT =====\n", result.stdout)
print("\n===== STDERR =====\n", result.stderr)
print("EXIT CODE:", result.returncode)
videos = glob.glob(f"{OUTPUT}/**/*.mp4", recursive=True)
print("VIDEOS:", videos)
if result.returncode != 0:
    raise RuntimeError(f"LTX generation failed with exit code {result.returncode}")
if not videos:
    raise RuntimeError("LTX finished but no MP4 was created.")
print("SUCCESS: ASTRAA first test shot generated.")


In [ ]:
import os, glob
videos=glob.glob('/content/LTX-Video/**/*.mp4', recursive=True)
print('\n'.join(videos[-10:]) if videos else 'No MP4 found yet.')


## Next
Once the first shot works, we will add reusable ASTRAA prompts and an extension workflow.